# Data audit และแผน 1 ชั่วโมง

เป้าหมาย: EDA ที่อ่านได้ใน notebook และ baseline submission ครบ 5 งาน ใช้ข้อมูลในเครื่อง ณ 2 พฤศจิกายน 2026 โค้ดใช้ชื่ออังกฤษ คำอธิบายภาษาไทย

## แผนการทำงาน

| เวลา | งาน | ผลลัพธ์ |
|---|---|---|
| 0–10 นาที | audit schema, label, image files และ sample submission | ข้อจำกัดและข้อมูลพร้อมใช้ |
| 10–25 นาที | EDA คุณภาพข้อมูล distribution และความสัมพันธ์ | 01_eda.ipynb ทั้ง 5 งาน |
| 25–45 นาที | train/validation และ refit baseline บน CPU | prediction ครบทุกงาน |
| 45–60 นาที | ตรวจไฟล์ อ่าน notebook outputs และสรุปข้อจำกัด | baseline_submission.csv |

ไม่ทำ hyperparameter search ในรอบแรก; การฝึกแยกอยู่ใน 02_baseline.ipynb ของแต่ละ task ข้อค้นพบทั้งหมดอยู่ใน notebook ไม่สร้างรายงาน findings แยก

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'sample_submission.csv').exists())
sys.path.insert(0, str(ROOT))
from shared.eda_utils import (
    TASKS, load_task, quality_table, outlier_table, bounds_table,
    association, strongest_pairs, period_changes, image_inventory, setup_plot,
)
setup_plot()

## ตรวจไฟล์และ label ทุกงาน

In [2]:
rows = []
test_ids = {}
for task in TASKS:
    train, test, target, base = load_task(ROOT, task)
    identifier = 'id' if task == 4 else 'player_id'
    test_ids[task] = test[identifier]
    rows.append({
        'task': task, 'train_rows': len(train) if train is not None else 0,
        'test_rows': len(test), 'test_columns': test.shape[1],
        'known_labels': int(train[target].notna().sum()) if train is not None else 0,
        'missing_train_labels': int(train[target].isna().sum()) if train is not None else 0,
        'duplicate_test_ids': int(test[identifier].duplicated().sum()),
        'train_test_id_overlap': len(set(train[identifier]) & set(test[identifier])) if train is not None else 0,
    })
audit = pd.DataFrame(rows)
display(audit)

,task,train_rows,test_rows,test_columns,known_labels,missing_train_labels,duplicate_test_ids,train_test_id_overlap
0,1,104000,25889,32,90816,13184,0,0
1,2,104000,25889,45,104000,0,0,0
2,3,104000,25889,33,104000,0,0,0
3,4,31546,25889,2,31546,0,0,0
4,5,0,25889,123,0,0,0,0


**Findings & Interpretation**

**Findings:** Test ทุกงานมี 25,889 แถว; known labels ของ Task 1–5 คือ [90816, 104000, 104000, 31546, 0]

**Interpretation:** Task 1 มี label หาย ส่วน Task 5 ไม่มี train; ขนาด test เท่ากันยังไม่แปลว่าใช้ player ID ลำดับเดียวกัน

**ML Implications:** แยก supervised training ตาม target ที่มีจริง และใช้ unsupervised baseline ใน Task 5

**Recommended Action:** ตรวจ sample schema และลำดับแถวก่อนรวมคำตอบ

## ตรวจ sample submission และ row mapping

In [3]:
sample = pd.read_csv(ROOT / 'sample_submission.csv')
display(sample.head(6))
display(sample.notna().sum().rename('non_missing').to_frame())
print('Task 1/2 same player_id at same row:', int(test_ids[1].eq(test_ids[2]).sum()))
print('Task 1/3 same player_id at same row:', int(test_ids[1].eq(test_ids[3]).sum()))
assert sample.columns.tolist() == ['id', 'task1', 'task2', 'task3', 'task4', 'task5']
assert sample.id.is_unique and len(sample) == 25889

Task 1/2 same player_id at same row: 0
Task 1/3 same player_id at same row: 0


,id,task1,task2,task3,task4,task5
0,ANS00001,1.0000,2.0000,"2,983.2000",0.0000,1.0000
1,ANS00002,0.0000,1.0000,529.0000,1.0000,1.0000
2,ANS00003,1.0000,3.0000,"15,682.4500",2.0000,0.0000
3,ANS00004,0.0000,2.0000,"7,200.0000",0.0000,1.0000
4,ANS00005,NaN,NaN,NaN,NaN,NaN
5,ANS00006,NaN,NaN,NaN,NaN,NaN


,non_missing
id,25889
task1,4
task2,4
task3,4
task4,4
task5,4


**Findings & Interpretation**

**Findings:** Sample ใช้ ANS ID และมีคำตอบตัวอย่างเพียง 4 แถวต่อ task; ไม่มีคอลัมน์เชื่อม ANS ID กับ player_id หรือ image id

**Interpretation:** ค่าตัวอย่างไม่ใช่ training label และลำดับ test ของแต่ละงานไม่ควรถูก sort หรือ merge เอง

**ML Implications:** Baseline ใช้แถวที่ i ของ test งานนั้นเป็นแถวที่ i ของ submission ตามสมมติฐาน positional mapping

**Recommended Action:** รักษา sample.id เดิมและยืนยันกติกา positional mapping กับผู้จัดก่อนอัปโหลด

## ตรวจข้อมูลภาพ

In [4]:
image_base = ROOT / TASKS[4][0]
image_train = pd.read_csv(image_base / 'train.csv')
image_test = pd.read_csv(image_base / 'test.csv')
rows = []
for split, data in [('train', image_train), ('test', image_test)]:
    exists = data.id.map(lambda image_id: (image_base / split / f'{image_id}.jpg').exists())
    rows.append({'split': split, 'manifest_rows': len(data), 'existing_images': exists.sum(),
                 'missing_images': (~exists).sum()})
display(pd.DataFrame(rows))

,split,manifest_rows,existing_images,missing_images
0,train,31546,2811,28735
1,test,25889,25889,0


**Findings & Interpretation**

**Findings:** ภาพฝึกมีไฟล์ 2,811 ภาพ จาก 31,546 รายการ; ภาพ test มีครบ 25,889 ภาพ

**Interpretation:** จำนวน label ใน CSV สูงกว่าจำนวนภาพที่ฝึกได้จริงมาก

**ML Implications:** กรองภาพอ่านได้ก่อน split โดยตรวจ full decode ใน Task 4 EDA

**Recommended Action:** สร้าง image baseline จาก subset พร้อมระบุ selection bias และกู้ไฟล์ในรอบถัดไป

## สรุปข้ามงานและขั้นถัดไป

| งาน | ประเด็นที่ยืนยัน | แนวทาง baseline |
|---|---|---|
| 1 Anti-cheat | known labels 90,816; หาย 13,184 | HistGradientBoosting, stratified holdout |
| 2 Segmentation | 4 คลาส มี numerical/categorical และ noise ตาม data card | categorical encoding + HistGradientBoosting |
| 3 Spending | มีค่า 0 จำนวนมากและหางขวายาว | HistGradientBoostingRegressor, เทียบ mean baseline |
| 4 Images | ภาพ train อ่านได้ 2,811/31,546 | pixel/histogram + ExtraTrees หรือ SVM |
| 5 Anomaly | ไม่มี train/label มี 4 ช่วงเวลา | delta/std + IsolationForest |

ข้อจำกัดร่วม: ยังไม่มี metric ทางการ, row mapping ที่ระบุโดยผู้จัด, timestamp และ provenance Task 5 ใช้ test เพื่อ fit แบบ transductive และ top 10% เป็นสมมติฐาน ต้องยืนยันกติกาก่อนส่ง ทุก EDA notebook มี findings อยู่ใต้การวิเคราะห์ และเก็บ output เมื่อรันสำเร็จ ไฟล์ submission อยู่ใน submissions/baseline_submission.csv หลังรัน baseline สำเร็จ ไม่มีการอัปโหลด Kaggle อัตโนมัติ